# Stage 4 — Модели

Два варианта из заметок к проекту:

| Вариант | Идея | Что на входе | Что на выходе |
|---|---|---|---|
| **A — FlattenMLP** | «Первые N кадров → длинный вектор → классификация изображений» | `(B, L, 1024+128)` → flatten → `(B, L·1152)` | `(B, N_CLASSES)` |
| **B — FrameRNN**  | «Рекуррентно подавать кадры на вход» — GRU по последовательности с pack_padded, аудио=0 если нет | `(B, L, 1152)` + длины | `(B, N_CLASSES)` |

Оба варианта едят один и тот же `FrameDataset` из stage 3.

In [1]:
# ============================================================
# 4.0 — Импорты и конфиг
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence
import json
from pathlib import Path

STAGE3_DIR = Path('../data2/frame_stage3')
STAGE4_DIR = Path('../data2/frame_stage4'); STAGE4_DIR.mkdir(exist_ok=True)

cfg = json.load(open(STAGE3_DIR / 'config.json'))
L         = cfg['L']
DIM_RGB   = cfg['dim_rgb']
DIM_AUD   = cfg['dim_audio']
N_CLASSES = cfg['n_classes']
GENRES    = cfg['genres']
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'L={L} dim_rgb={DIM_RGB} dim_aud={DIM_AUD} n_classes={N_CLASSES} device={DEVICE}')

L=60 dim_rgb=1024 dim_aud=128 n_classes=12 device=cuda


In [2]:
# ============================================================
# 4.1 — Вариант A: FlattenMLP
# ============================================================
# Берём первые L кадров (N_FRAMES_USE ≤ L), конкатенируем rgb+audio
# для каждого кадра, сплющиваем → MLP. Простейший подход из заметок:
# "взять первые 10 кадров (с промежутком), создать длинный входной вектор
#  и будет задача классификация изображений".
#
# Для кадров где их нет — zero-padding оставлен в Dataset, MLP учится
# игнорировать нули.

class FlattenMLP(nn.Module):
    def __init__(self, n_frames, dim_rgb, dim_aud, n_classes,
                 hidden=(1024, 512), dropout=0.4):
        super().__init__()
        self.n_frames = n_frames
        in_dim = n_frames * (dim_rgb + dim_aud)

        layers = []
        prev = in_dim
        for h in hidden:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.GELU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, rgb, aud, lengths=None):
        # rgb: (B, L, Dr)  aud: (B, L, Da)
        K = self.n_frames
        x = torch.cat([rgb[:, :K], aud[:, :K]], dim=-1)   # (B, K, Dr+Da)
        x = x.reshape(x.size(0), -1)                      # (B, K*(Dr+Da))
        return self.net(x)


# quick sanity
m = FlattenMLP(n_frames=30, dim_rgb=DIM_RGB, dim_aud=DIM_AUD, n_classes=N_CLASSES)
nparams = sum(p.numel() for p in m.parameters())
print(f'FlattenMLP (n_frames=30): {nparams/1e6:.2f}M params')
r = torch.randn(4, L, DIM_RGB); a = torch.randn(4, L, DIM_AUD)
print(f'  forward ok → {tuple(m(r, a).shape)}')

FlattenMLP (n_frames=30): 35.92M params
  forward ok → (4, 12)


In [3]:
# ============================================================
# 4.2 — Вариант B: FrameRNN (GRU + pack_padded)
# ============================================================
# "Кусок аудио и кусок видео на вход каждый шаг в рекуррентную сеть.
#  Отсутствие аудио подаёт нули. Нет жёсткой привязки к количеству
#  кадров" → pack_padded_sequence маскирует паддинг.

class FrameRNN(nn.Module):
    def __init__(self, dim_rgb, dim_aud, n_classes,
                 proj_dim=256, rnn_hidden=256, rnn_layers=1,
                 bidirectional=True, dropout=0.3, cell='gru'):
        super().__init__()
        # Проекция каждого кадра (Dr+Da → proj_dim) чтобы RNN не ел
        # сырой 1152-мерный вход
        self.proj = nn.Sequential(
            nn.Linear(dim_rgb + dim_aud, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        RNN = {'gru': nn.GRU, 'lstm': nn.LSTM}[cell]
        self.rnn = RNN(
            input_size=proj_dim,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if rnn_layers > 1 else 0.0,
        )
        out_dim = rnn_hidden * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Dropout(dropout),
            nn.Linear(out_dim, n_classes),
        )
        self.bidirectional = bidirectional
        self.cell = cell

    def forward(self, rgb, aud, lengths):
        # rgb: (B, L, Dr), aud: (B, L, Da), lengths: (B,)
        x = torch.cat([rgb, aud], dim=-1)          # (B, L, Dr+Da)
        x = self.proj(x)                           # (B, L, proj_dim)

        # pack — RNN не тратит вычисления на паддинг
        lens_cpu = lengths.detach().cpu().clamp(min=1)
        packed = pack_padded_sequence(
            x, lens_cpu, batch_first=True, enforce_sorted=False
        )
        out, hidden = self.rnn(packed)

        if self.cell == 'lstm':
            hidden = hidden[0]   # (h, c) → берём h

        # hidden: (num_layers*num_dirs, B, H); берём последний слой
        if self.bidirectional:
            h = torch.cat([hidden[-2], hidden[-1]], dim=-1)   # (B, 2H)
        else:
            h = hidden[-1]                                    # (B, H)

        return self.head(h)


# sanity
m = FrameRNN(DIM_RGB, DIM_AUD, N_CLASSES, proj_dim=256, rnn_hidden=256)
nparams = sum(p.numel() for p in m.parameters())
print(f'FrameRNN (GRU, proj=256, hid=256, bi): {nparams/1e6:.2f}M params')
r = torch.randn(4, L, DIM_RGB); a = torch.randn(4, L, DIM_AUD)
lens = torch.tensor([L, L-5, L-20, 3])
print(f'  forward ok → {tuple(m(r, a, lens).shape)}')

FrameRNN (GRU, proj=256, hid=256, bi): 1.09M params
  forward ok → (4, 12)


In [4]:
# ============================================================
# 4.3 — Фабрика моделей + параметры для обучения
# ============================================================
def build_model(kind: str):
    if kind == 'flatten':
        return FlattenMLP(
            n_frames=min(30, L), dim_rgb=DIM_RGB, dim_aud=DIM_AUD,
            n_classes=N_CLASSES, hidden=(1024, 512), dropout=0.4,
        )
    if kind == 'rnn':
        return FrameRNN(
            dim_rgb=DIM_RGB, dim_aud=DIM_AUD, n_classes=N_CLASSES,
            proj_dim=256, rnn_hidden=256, rnn_layers=1,
            bidirectional=True, dropout=0.3, cell='gru',
        )
    raise ValueError(kind)

# сохраняем инициализации (чтобы stage5 стартовал с одной точки)
for kind in ('flatten', 'rnn'):
    torch.save(build_model(kind).state_dict(), STAGE4_DIR / f'init_{kind}.pt')
    print(f'  saved init_{kind}.pt')

train_cfg = {
    'variants'  : ['flatten', 'rnn'],
    'batch_size': 256,
    'epochs'    : 25,
    'lr'        : 3e-4,
    'wd'        : 1e-4,
    'label_smooth': 0.05,
    'sched'     : 'cosine',
    'seed'      : 42,
}
with open(STAGE4_DIR / 'train_config.json', 'w') as f:
    json.dump(train_cfg, f, indent=2)
print(f'train_config saved')

  saved init_flatten.pt
  saved init_rnn.pt
train_config saved


In [ ]:
from clearml import Task
import json
from pathlib import Path

task = Task.init(project_name="YT8M", task_name="04_models")

STAGE4_DIR = Path('../data2/frame_stage4')

train_cfg = json.load(open(STAGE4_DIR / 'train_config.json'))
task.connect(train_cfg, name="train_config")

# Параметры моделей (из build_model)
model_hparams = {
    "flatten": {"hidden": [1024, 512], "dropout": 0.4, "n_frames_use": 30},
    "rnn":     {"proj_dim": 256, "rnn_hidden": 256, "rnn_layers": 1,
                "bidirectional": True, "dropout": 0.3, "cell": "gru"},
}
task.connect(model_hparams, name="model_hparams")

# Артефакты — веса инициализации и конфиг
task.upload_artifact('train_config_json', artifact_object=str(STAGE4_DIR / 'train_config.json'))
for pt in sorted(STAGE4_DIR.glob('init_*.pt')):
    task.upload_artifact(f'init_weights_{pt.stem}', artifact_object=str(pt))

task.close()
print("ClearML task closed: 04_models")
